In [7]:
import pandas as pd
import numpy as np
import pickle
import re

from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# Load dataset
df = pd.read_csv('dataset/email_spam_100k.csv')


# Clean text
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\W+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['body'] = df['body'].apply(clean_text)

In [8]:
X = df['body']
y = df['label']

# Split
X_train1, X_test1, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# TF-IDF
vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 3),     
    max_df=0.9,
    min_df=2,             
    sublinear_tf=True,      
    max_features=40000     
)

X_train = vectorizer.fit_transform(X_train1)
X_test = vectorizer.transform(X_test1)

In [9]:

# Model
model = LinearSVC(C=1.5)

model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)



In [10]:
# Evaluate
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print(classification_report(y_test, y_pred))

Accuracy: 0.9765
Precision: 0.971
              precision    recall  f1-score   support

           0       0.98      0.97      0.98     10443
           1       0.97      0.98      0.98      9557

    accuracy                           0.98     20000
   macro avg       0.98      0.98      0.98     20000
weighted avg       0.98      0.98      0.98     20000



In [11]:
# Save
pickle.dump(model, open("svc.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))